# 따릉이 스테이션 별 수요도 예측 Baseline

## 환경 설정

In [1]:
import sys
import os

# ==========================================
# 프로젝트 경로 및 라이브러리 설정
# ==========================================
# 상위 폴더(프로젝트 루트)를 모듈 검색 경로에 추가
sys.path.append(os.path.dirname(os.getcwd()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path

# Scikit-learn 모델 및 유틸리티
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, VotingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

# 모델링 및 튜닝 도구
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import optuna

# ==========================================
# 시각화 및 환경 설정
# ==========================================
# 그래프에서 음수 부호(-) 깨짐 방지
plt.rcParams["axes.unicode_minus"] = False
# 한글 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'

# 난수 시드 고정
SEED = 42
np.random.seed(SEED)

print("\n========== 데이터 분석 환경 설정 완료 ==========")
print(f"설정된 시드 값: {SEED}")
print("라이브러리 로드 완료")


========== 데이터 분석 환경 설정 완료 ==========
설정된 시드 값: 42
라이브러리 로드 완료


## 데이터베이스 연결 및 데이터 조회

In [2]:
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from functools import reduce

# ==========================================
# 환경 설정 및 DB 연결
# ==========================================
load_dotenv()

DB_USER     = os.getenv("DB_USER", "root")
DB_PASSWORD = os.getenv("DB_PASSWORD", "password")
DB_HOST     = os.getenv("DB_HOST", "localhost")
DB_PORT     = os.getenv("DB_PORT", "3306")
DB_NAME     = os.getenv("DB_NAME", "seoul_bike")

DATABASE_URL = f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(DATABASE_URL)

# 데이터베이스 연결 확인
with engine.connect() as conn:
    if conn.execute(text("SELECT 1")).scalar() == 1:
        print("\n========== 데이터베이스 연결 성공 ==========")
    else:
        print("\n========== 데이터베이스 연결 실패 ==========")


# ==========================================
# 데이터베이스 테이블 로드 (DataFrame)
# ==========================================
print("========== 각 테이블 데이터 로드 시작 ==========")

# 환경/기상 데이터
hourly_air_2024_df    = pd.read_sql_table("hourly_air_2024", con=engine)
hourly_precip_2024_df = pd.read_sql_table("hourly_precip_2024", con=engine)
hourly_snow_2024_df   = pd.read_sql_table("hourly_snow_2024", con=engine)
hourly_temp_2024_df   = pd.read_sql_table("hourly_temp_2024", con=engine)
rt_air_df             = pd.read_sql_table("rt_air", con=engine)
rt_weather_df         = pd.read_sql_table("rt_weather", con=engine)

# 인프라 데이터
infra_business_df     = pd.read_sql_table("infra_business", con=engine)
infra_park_df         = pd.read_sql_table("infra_park", con=engine)
infra_river_df        = pd.read_sql_table("infra_river", con=engine)
infra_school_df       = pd.read_sql_table("infra_school", con=engine)
infra_univ_df         = pd.read_sql_table("infra_univ", con=engine)

# 인구 데이터
pop_flow_2024_df      = pd.read_sql_table("pop_flow_2024", con=engine)
pop_living_2024_df    = pd.read_sql_table("pop_living_2024", con=engine)

# 따릉이 및 기타 데이터
rent_history_df       = pd.read_sql_table("rent_history", con=engine)
korea_holidays_df     = pd.read_sql_table("korea_holidays", con=engine)
station_loc_df        = pd.read_sql_table("station_loc", con=engine)
rt_bike_status_df     = pd.read_sql_table("rt_bike_status", con=engine)

print("========== 모든 테이블 데이터 로드 완료 ==========")


========== 데이터베이스 연결 성공 ==========
========== 각 테이블 데이터 로드 시작 ==========
========== 모든 테이블 데이터 로드 완료 ==========


In [3]:
# 데이터프레임.info()
# 데이터프레임.head()

## 전처리

In [ ]:
# ==========================================
# 기상 데이터 전처리 함수
# ==========================================
def preprocess_weather_df(df, col_name):
    """
    기상 데이터의 이상치 제거, 날짜 변환 및 1시간 단위 리샘플링 수행
    """
    df = df.copy()

    # 불필요한 ID 컬럼 제거
    if 'id' in df.columns:
        df = df.drop(columns=['id'])

    # 날짜 변환 및 이상치(-9) 처리
    df['measure_date'] = pd.to_datetime(df['measure_date'])
    df.replace([-9, -9.0], np.nan, inplace=True)

    # 1시간 단위 리샘플링 후 평균값 계산
    df_resampled = (
        df.set_index('measure_date')
        .groupby('region_name')[col_name]
        .resample('1h')
        .mean()
        .reset_index()
    )

    return df_resampled


# ==========================================
# 데이터 전처리 실행
# ==========================================
print("\n========== 기상 데이터 전처리 시작 ==========")

processed_air    = preprocess_weather_df(hourly_air_2024_df, 'pm10')
processed_temp   = preprocess_weather_df(hourly_temp_2024_df, 'temperature')
processed_precip = preprocess_weather_df(hourly_precip_2024_df, 'precipitation')
processed_snow   = preprocess_weather_df(hourly_snow_2024_df, 'snowfall')

print("기상 데이터별 전처리 완료")


# ==========================================
# 최종 데이터 통합(Merge)
# ==========================================
print("========== 기상 데이터 통합 시작 ==========")

weather_dfs = [processed_air, processed_temp, processed_precip, processed_snow]

# reduce를 활용한 외부 조인(outer join) 수행
env_master_2024_df = reduce(
    lambda left, right: pd.merge(left, right, on=['measure_date', 'region_name'], how='outer'),
    weather_dfs
)

# 최종 결측치 처리
env_master_2024_df['precipitation'] = env_master_2024_df['precipitation'].fillna(0)
env_master_2024_df['snowfall'] = env_master_2024_df['snowfall'].fillna(0)

print("========== 기상 데이터 통합 종료 ==========")

In [ ]:
# ==========================================
# DB(SQL) 저장 로직
# ==========================================
table_name = 'env_master_2024'
env_master_2024_df.to_sql(
    name=table_name,
    con=engine,
    if_exists='replace',
    index=False
)
print(f"========== DB 적재 성공 ==========")

In [4]:
# ==========================================
# 생활인구 데이터 전처리
# ==========================================
print("\n========== 생활인구 데이터 전처리 시작 ==========")
district_map = {'11500': '강서구', '11560': '영등포구', '11440': '마포구', '11710': '송파구'}
pop_living_2024_df['district_name'] = pop_living_2024_df['adstrd_code_se'].map(district_map)

# 컬럼명 정리
pop_living_2024_df.rename(columns={'tot_lvpop_co': 'lvgpop_tot'}, inplace=True)

# 연령대별 합산
pop_living_2024_df['lvgpop_10s'] = pop_living_2024_df[['male_f10t14_lvpop_co', 'male_f15t19_lvpop_co', 'female_f10t14_lvpop_co', 'female_f15t19_lvpop_co']].sum(axis=1)
pop_living_2024_df['lvgpop_20s'] = pop_living_2024_df[['male_f20t24_lvpop_co', 'male_f25t29_lvpop_co', 'female_f20t24_lvpop_co', 'female_f25t29_lvpop_co']].sum(axis=1)
pop_living_2024_df['lvgpop_30s'] = pop_living_2024_df[['male_f30t34_lvpop_co', 'male_f35t39_lvpop_co', 'female_f30t34_lvpop_co', 'female_f35t39_lvpop_co']].sum(axis=1)
pop_living_2024_df['lvgpop_40s'] = pop_living_2024_df[['male_f40t44_lvpop_co', 'male_f45t49_lvpop_co', 'female_f40t44_lvpop_co', 'female_f45t49_lvpop_co']].sum(axis=1)
pop_living_2024_df['lvgpop_50s'] = pop_living_2024_df[['male_f50t54_lvpop_co', 'male_f55t59_lvpop_co', 'female_f50t54_lvpop_co', 'female_f55t59_lvpop_co']].sum(axis=1)
pop_living_2024_df['lvgpop_60up'] = pop_living_2024_df[['male_f60t64_lvpop_co', 'male_f65t69_lvpop_co', 'male_f70t74_lvpop_co', 'female_f60t64_lvpop_co', 'female_f65t69_lvpop_co', 'female_f70t74_lvpop_co']].sum(axis=1)

# 병합을 위한 키(Key) 생성
pop_living_2024_df = pop_living_2024_df[['stdr_de_id', 'tmzon_pd_se', 'district_name', 'lvgpop_tot', 'lvgpop_10s', 'lvgpop_20s', 'lvgpop_30s', 'lvgpop_40s', 'lvgpop_50s', 'lvgpop_60up']].copy()
pop_living_2024_df['date_str'] = pop_living_2024_df['stdr_de_id'].astype(str)
pop_living_2024_df['hour_str'] = pop_living_2024_df['tmzon_pd_se'].astype(str).str.zfill(2)


# ==========================================
# 2024년 인구 통합 마스터 생성
# ==========================================
print("========== 인구 통합 마스터 생성 시작 ==========")
date_rng = pd.date_range(start='2024-01-01 00:00:00', end='2024-12-31 23:00:00', freq='h')
districts = ['강서구', '영등포구', '마포구', '송파구']

pop_master_2024_list = []
for dist in districts:
    df_temp = pd.DataFrame(date_rng, columns=['datetime'])
    df_temp['district_name'] = dist
    df_temp['date_str'] = df_temp['datetime'].dt.strftime('%Y%m%d')
    df_temp['hour_str'] = df_temp['datetime'].dt.strftime('%H')
    df_temp['weekday'] = df_temp['datetime'].dt.weekday
    df_temp['quarter_str'] = '2024' + df_temp['datetime'].dt.quarter.astype(str)
    pop_master_2024_list.append(df_temp)

pop_master_2024_df = pd.concat(pop_master_2024_list, ignore_index=True)


# ==========================================
# 유동인구 분배 및 병합 로직
# ==========================================
print("========== 유동인구 데이터 병합 및 계산 시작 ==========")
pop_flow_2024_df = pd.merge(
    pop_master_2024_df,
    pop_flow_2024_df,
    left_on=['quarter_str', 'district_name'],
    right_on=['stdr_yyqu_cd', 'signgu_cd_nm'],
    how='left'
)

# 분배 기준값 설정
time_divisors = {
    '00': 6, '01': 6, '02': 6, '03': 6, '04': 6, '05': 6,
    '06': 5, '07': 5, '08': 5, '09': 5, '10': 5,
    '11': 3, '12': 3, '13': 3, '14': 3, '15': 3, '16': 3,
    '17': 4, '18': 4, '19': 4, '20': 4,
    '21': 3, '22': 3, '23': 3
}
weekday_col_map = {
    0: 'mon_flpop_co', 1: 'tues_flpop_co', 2: 'wed_flpop_co',
    3: 'thur_flpop_co', 4: 'fri_flpop_co', 5: 'sat_flpop_co', 6: 'sun_flpop_co'
}

def calc_flow_pop(row):
    """시간대별 유동인구 분배 계산 함수"""
    h, wd = row['hour_str'], row['weekday']
    div = time_divisors.get(h, 1)

    # 시간대별 카테고리 매핑
    if h in ['00','01','02','03','04','05']: time_pop = row['tmzon_00_06_flpop_co']
    elif h in ['06','07','08','09','10']: time_pop = row['tmzon_06_11_flpop_co']
    elif h in ['11','12','13']: time_pop = row['tmzon_11_14_flpop_co']
    elif h in ['14','15','16']: time_pop = row['tmzon_14_17_flpop_co']
    elif h in ['17','18','19','20']: time_pop = row['tmzon_17_21_flpop_co']
    else: time_pop = row['tmzon_21_24_flpop_co']

    day_pop = row[weekday_col_map[wd]]
    tot_pop = row['tot_flpop_co']

    if pd.notna(tot_pop) and tot_pop > 0:
        est_tot_flwpop = (time_pop / div) * (day_pop / tot_pop) / 13
        return pd.Series([
            est_tot_flwpop,
            est_tot_flwpop * (row['agrde_10_flpop_co'] / tot_pop),
            est_tot_flwpop * (row['agrde_20_flpop_co'] / tot_pop),
            est_tot_flwpop * (row['agrde_30_flpop_co'] / tot_pop),
            est_tot_flwpop * (row['agrde_40_flpop_co'] / tot_pop),
            est_tot_flwpop * (row['agrde_50_flpop_co'] / tot_pop),
            est_tot_flwpop * (row['agrde_60_above_flpop_co'] / tot_pop)
        ])
    return pd.Series([0, 0, 0, 0, 0, 0, 0])

# 유동인구 계산 적용
pop_flow_2024_df[['flwpop_tot', 'flwpop_10s', 'flwpop_20s', 'flwpop_30s', 'flwpop_40s', 'flwpop_50s', 'flwpop_60up']] = \
    pop_flow_2024_df.apply(calc_flow_pop, axis=1)


# ==========================================
# 최종 병합 및 정리
# ==========================================
print("========== 최종 데이터 병합 완료 ==========")
pop_master_2024_df = pd.merge(
    pop_flow_2024_df,
    pop_living_2024_df,
    on=['date_str', 'hour_str', 'district_name'],
    how='left'
)

# 핵심 컬럼만 추출
pop_master_2024_cols = [
    'datetime', 'district_name',
    'flwpop_tot', 'flwpop_10s', 'flwpop_20s', 'flwpop_30s', 'flwpop_40s', 'flwpop_50s', 'flwpop_60up',
    'lvgpop_tot', 'lvgpop_10s', 'lvgpop_20s', 'lvgpop_30s', 'lvgpop_40s', 'lvgpop_50s', 'lvgpop_60up'
]
pop_master_2024_df = pop_master_2024_df[pop_master_2024_cols]

# 결과 확인
print("\n[데이터 요약 정보]")
print(pop_master_2024_df.info())
print("\n[데이터 상단 5행]")
print(pop_master_2024_df.head())


========== 생활인구 데이터 전처리 시작 ==========
========== 인구 통합 마스터 생성 시작 ==========
========== 유동인구 데이터 병합 및 계산 시작 ==========
========== 최종 데이터 병합 완료 ==========

[데이터 요약 정보]
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35136 entries, 0 to 35135
Data columns (total 16 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   datetime       35136 non-null  datetime64[ns]
 1   district_name  35136 non-null  object        
 2   flwpop_tot     35136 non-null  float64       
 3   flwpop_10s     35136 non-null  float64       
 4   flwpop_20s     35136 non-null  float64       
 5   flwpop_30s     35136 non-null  float64       
 6   flwpop_40s     35136 non-null  float64       
 7   flwpop_50s     35136 non-null  float64       
 8   flwpop_60up    35136 non-null  float64       
 9   lvgpop_tot     35136 non-null  float64       
 10  lvgpop_10s     35136 non-null  float64       
 11  lvgpop_20s     35136 non-null  float64       
 12  lvgpo

In [5]:
# ==========================================
# DB(SQL) 저장 로직
# ==========================================
table_name = 'pop_master_2024'
pop_master_2024_df.to_sql(
    name=table_name,
    con=engine,
    if_exists='replace',
    index=False
)
print(f"========== DB 적재 성공 ==========")

========== DB 적재 성공 ==========


## EDA

## 피처(X) / 타깃(Y) 분리

## Train / Validation / Test 3분할

## 평가 지표 함수 및 기본 모델 학습

## 여러 모델 비교(Ridge, RandomForest, XGBoost, LightGBM)

## 앙상블(Voting Regressor)

## 최종 모델 선택

## Test셋 최종 평가

## 모델 저장(pkl) 및 저장된 모델 검증